# Task 1b — 证据抽取（BIO token分类）baseline
IEEE BigData 2026 Explainable Suicide Risk Detection

从帖子里抽取支撑风险判断的证据短语(span)。做法：token级BIO标注 + 微调。
评测：Phrase F1（官方规则：含/被含算对，长度≤gold的3倍，大小写不敏感）。
indicator帖子95%证据为none → 规则上直接输出空，简化任务。

**运行前**：GPU；Drive里有最新 common.py + train_clean.csv。


## 0. 路径与超参


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/IEEE_BigData2026"   # <<<< 改路径
MODEL_TAG   = "deproberta"

MAX_LEN     = 512
BATCH_SIZE  = 8
EPOCHS      = 4
LR          = 2e-5
SEED        = 42


## 1. 挂载Drive+依赖


In [ ]:
import os, sys
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception: print("非Colab")
assert os.path.isdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR); os.chdir(PROJECT_DIR)
print("工作目录:", os.getcwd())
import importlib, subprocess
def ensure(p,i=None):
    try: importlib.import_module(i or p)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
for p,i in [("sentencepiece","sentencepiece"),("accelerate","accelerate"),("seqeval","seqeval")]:
    ensure(p,i)
import torch
assert torch.cuda.is_available(), "选GPU！"
print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
工作目录: /content/drive/MyDrive/IEEE_BigData2026
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. 导入地基+数据


In [ ]:
import numpy as np, pandas as pd
from common import load_data, get_fold, MODELS, N_FOLDS
df = load_data()
print(f"数据 {len(df)} 行")

def set_seed(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)


数据 1635 行


## 3. BIO标注：把evidence对齐到token
用offset_mapping把字符级证据位置映射到token级 B/I/O 标签。
标签: 0=O(非证据), 1=B(证据开头), 2=I(证据内部)


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODELS[MODEL_TAG], use_fast=True)

def char_evidence_mask(post, evidence_str):
    """字符级：每个字符是否属于证据(verbatim匹配，none/找不到→全0)"""
    post_l = post.lower(); mask=[0]*len(post)
    if isinstance(evidence_str,str):
        for sp in evidence_str.split(';'):
            sp=sp.strip()
            if not sp or sp.lower()=='none': continue
            spl=sp.lower(); start=0
            while True:
                idx=post_l.find(spl,start)
                if idx<0: break
                for k in range(idx,idx+len(spl)): mask[k]=1
                start=idx+len(spl)
    return mask

def encode_with_labels(post, evidence_str, max_len):
    """返回tokenizer编码 + token级BIO标签"""
    cmask = char_evidence_mask(post, evidence_str)
    enc = tokenizer(post, truncation=True, max_length=max_len,
                    padding="max_length", return_offsets_mapping=True,
                    return_tensors="pt")
    offsets = enc["offset_mapping"][0].tolist()
    labels=[]
    prev=0
    for (s,e) in offsets:
        if s==e:  # 特殊token/padding
            labels.append(-100)  # 损失忽略
        else:
            is_ev = any(cmask[s:e]) if e<=len(cmask) else False
            if is_ev:
                labels.append(1 if prev==0 else 2)  # B 或 I
                prev=1
            else:
                labels.append(0); prev=0
    enc.pop("offset_mapping")
    return {k:v.squeeze(0) for k,v in enc.items()}, torch.tensor(labels)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

## 4. Dataset


In [ ]:
from torch.utils.data import Dataset
class EvidenceDataset(Dataset):
    def __init__(self, df_sub, max_len):
        self.posts=df_sub["post"].astype(str).tolist()
        self.evs=df_sub["evidence"].astype(str).tolist()
        self.risk=df_sub["risk_level"].tolist()
        self.max_len=max_len
    def __len__(self): return len(self.posts)
    def __getitem__(self, i):
        enc, labels = encode_with_labels(self.posts[i], self.evs[i], self.max_len)
        enc["labels"]=labels
        return enc


## 5. Phrase F1 评测（官方规则）


In [ ]:
def phrase_f1_single(pred_spans, gold_spans):
    pred=[p.strip().lower() for p in pred_spans if p.strip()]
    gold=[g.strip().lower() for g in gold_spans if g.strip() and g.strip().lower()!='none']
    if len(gold)==0 and len(pred)==0: return 1.0,1.0,1.0
    if len(gold)==0 or len(pred)==0: return 0.0,0.0,0.0
    def match(p,g):
        if len(p.split())>3*len(g.split()): return False
        return (g in p) or (p in g)
    tp_p=sum(1 for p in pred if any(match(p,g) for g in gold))
    tp_r=sum(1 for g in gold if any(match(p,g) for p in pred))
    prec=tp_p/len(pred); rec=tp_r/len(gold)
    f1=2*prec*rec/(prec+rec) if prec+rec>0 else 0
    return prec,rec,f1

def decode_spans(post, pred_labels, offsets):
    """从token级BIO预测还原出文本span"""
    spans=[]; cur_start=None
    for (s,e),lab in zip(offsets, pred_labels):
        if s==e: continue
        if lab in (1,2):  # B或I
            if cur_start is None: cur_start=s
            cur_end=e
        else:
            if cur_start is not None: spans.append(post[cur_start:cur_end]); cur_start=None
    if cur_start is not None: spans.append(post[cur_start:cur_end])
    return spans


## 6. 单fold训练+预测


In [ ]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments

def train_one_fold(fold, df):
    tr_df, va_df = get_fold(df, fold)
    model=AutoModelForTokenClassification.from_pretrained(MODELS[MODEL_TAG], num_labels=3)
    model.to("cuda")
    tr_ds=EvidenceDataset(tr_df, MAX_LEN)
    va_ds=EvidenceDataset(va_df, MAX_LEN)
    use_bf16=torch.cuda.is_bf16_supported()
    args=TrainingArguments(
        output_dir=f"/content/ckpt_t1b_f{fold}", num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=16,
        learning_rate=LR, eval_strategy="no", save_strategy="no", logging_steps=50,
        warmup_ratio=0.1, weight_decay=0.01,
        bf16=use_bf16, fp16=not use_bf16, report_to="none", seed=SEED,
    )
    trainer=Trainer(model=model, args=args, train_dataset=tr_ds)
    trainer.train()

    # 预测+算该fold的Phrase F1
    model.eval()
    f1s=[]
    for _, row in va_df.iterrows():
        post=str(row["post"])
        # indicator规则：直接空证据
        if row["risk_level"]=="indicator":
            pred_spans=[]
        else:
            enc=tokenizer(post, truncation=True, max_length=MAX_LEN, padding="max_length",
                          return_offsets_mapping=True, return_tensors="pt")
            offsets=enc["offset_mapping"][0].tolist()
            with torch.no_grad():
                inp={k:v.to("cuda") for k,v in enc.items() if k!="offset_mapping"}
                logits=model(**inp).logits[0]
            preds=logits.argmax(-1).cpu().tolist()
            pred_spans=decode_spans(post, preds, offsets)
        gold_spans=str(row["evidence"]).split(';')
        _,_,f1=phrase_f1_single(pred_spans, gold_spans)
        f1s.append(f1)
    del model, trainer; torch.cuda.empty_cache()
    return np.mean(f1s), va_df.index.tolist(), f1s


## 7. 跑满5fold


In [ ]:
oof_f1=np.zeros(len(df))
fold_scores=[]
for fold in range(N_FOLDS):
    print(f"\n{'='*50}\n  Task1b Fold {fold} / {MODEL_TAG}\n{'='*50}")
    mean_f1, idx, f1s = train_one_fold(fold, df)
    oof_f1[idx]=f1s
    fold_scores.append(mean_f1)
    print(f"  >> Fold {fold} Phrase F1 = {mean_f1:.4f}")

overall=np.mean(fold_scores)   # 直接平均各fold（每fold样本数接近，无偏）
print(f"\n★ Task1b 整体 OOF Phrase F1 = {overall:.4f}")
print(f"  各fold: {[f'{s:.4f}' for s in fold_scores]}")



  Task1b Fold 0 / deproberta


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: rafalposwiata/deproberta-large-v1
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,0.388990
100,0.185766
150,0.160151
200,0.147536
250,0.127403
300,0.116642
350,0.093459
400,0.083459
450,0.069385
500,0.068845


...

  >> Fold 3 Phrase F1 = 0.8164

  Task1b Fold 4 / deproberta


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: rafalposwiata/deproberta-large-v1
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,0.497561
100,0.186409
150,0.168467
200,0.139591
250,0.128487
300,0.111191
350,0.076018
400,0.075433
450,0.071173
500,0.073242


  >> Fold 4 Phrase F1 = 0.7986

★ Task1b 整体 OOF Phrase F1 = 0.1617
  各fold: ['0.8008', '0.8130', '0.8643', '0.8164', '0.7986']


## 8. 保存分数


In [ ]:
RESULTS_DIR=os.path.join(PROJECT_DIR,"results"); os.makedirs(RESULTS_DIR, exist_ok=True)
sp=os.path.join(RESULTS_DIR,"scores_t1b.csv")
row={"model":MODEL_TAG,"phrase_f1":round(float(overall),4),
     "folds":";".join(f"{s:.4f}" for s in fold_scores)}
if os.path.exists(sp):
    sdf=pd.read_csv(sp); sdf=sdf[sdf["model"]!=MODEL_TAG]
    sdf=pd.concat([sdf,pd.DataFrame([row])],ignore_index=True)
else: sdf=pd.DataFrame([row])
sdf.to_csv(sp,index=False)
print(f"✓ 分数已存: results/scores_t1b.csv")
print(sdf.to_string(index=False))
print(f"\nTask1b baseline完成。Phrase F1 = {overall:.4f}")
print("下一步可做：LLM抽取路线 对比 / 处理那173个改写的证据")


✓ 分数已存: results/scores_t1b.csv
     model  phrase_f1                              folds
deproberta     0.1617 0.8008;0.8130;0.8643;0.8164;0.7986

Task1b baseline完成。Phrase F1 = 0.1617
下一步可做：LLM抽取路线 对比 / 处理那173个改写的证据
